# Advanced Ray Train + Ray Data: Removing Training Bottlenecks with Async Checkpointing, Async Validation, and Mid-Epoch Resumption

In this tutorial, you build on the basics of Ray Train to learn how to **remove common performance and reliability bottlenecks in distributed deep learning training**. Instead of optimizing for model accuracy, you focus on keeping GPUs busy, minimizing wasted work, and making training resilient to failures at scale.

In typical distributed training systems, GPUs often sit idle because of:
- **Synchronous checkpointing**, where training blocks while checkpoints upload to remote storage.
- **Synchronous validation**, where expensive training GPUs stop making progress while validation runs.
- **Failure recovery**, where a worker restart forces you to reprocess large portions of already-seen data.

Ray Train addresses these issues with three advanced features, all of which you implement end to end in this notebook.

## What You will Learn

### 1. Asynchronous Checkpointing
You upload checkpoints to persistent storage **in the background**, allowing training to continue uninterrupted. This removes I/O stalls from the critical training path. For more details, see the Ray Train [checkpointing documentation](https://docs.ray.io/en/latest/train/user-guides/checkpoints.html).

### 2. Asynchronous Validation
You run validation out of band in separate Ray Trainers or with Ray Data batch inference (Ray actors). Because validation typically requires less compute than gradient-based training, you can move it to cheaper or smaller GPUs and allow training to continue while validation runs. For more details, see the Ray Train [validation documentation](https://docs.ray.io/en/latest/train/user-guides/asynchronous-validation.html)

### 3. Mid-Epoch Resumption
You recover from failures **without restarting the epoch** by checkpointing and restoring not only model and optimizer state, but also the **Ray Data iterator state**. This enables true mid-epoch recovery and avoids wasted computation. For more details, see the Anyscale [mid-epoch resumption documentation](https://docs.anyscale.com/runtime/mid-epoch-resumption). Please note, that this feature is specific to Anyscale Runtime, and is not available with Ray OSS.

## Why This Matters

At scale, even small inefficiencies add up:
- Blocking checkpoint uploads and validation significantly reduce GPU utilization.
- Restarting epochs after failures wastes time, money, and energy.
- Validation often requires less compute than training, yet traditionally runs on the same expensive hardware.

By combining Ray Train and Ray Data, you learn how to:
- Overlap training with checkpoint uploads and validation.
- Use heterogeneous hardware efficiently.
- Recover quickly from failures with minimal lost progress.

## Tutorial Overview

In this notebook, you:
1. Prepare a deterministic MNIST dataset using Ray Data.
2. Train a ResNet-based model with Ray Train and DDP.
3. Enable **asynchronous checkpoint uploads** during training.
4. Launch **asynchronous validation runs** triggered by checkpoints.
5. Implement **mid-epoch checkpointing and resumption** using dataset iterator state.
6. Inspect training and validation metrics produced asynchronously.
7. Load the final checkpoint and run inference.

This tutorial is designed for readers who are already familiar with basic Ray Train concepts and want to push distributed training systems toward **higher utilization, lower cost, and stronger fault tolerance**. If you have not done so already, this [introductory Ray Train tutorial](https://console.anyscale.com/template-preview/ai-libraries-pytorch?file=%252Ffiles%252F2_Intro_Train.ipynb) covers all prerequisits needed for this lesson.

## Imports

You import the core Ray Train APIs you need to demonstrate the *new* features in this tutorial. In addition to standard PyTorch components, you bring in:

- **`CheckpointUploadMode`** to enable **asynchronous checkpoint uploads**, so training can continue while checkpoints upload to persistent storage.
- **`TorchTrainer`** and related **Ray Train** configuration objects (`ScalingConfig`, `RunConfig`, `FailureConfig`) to run distributed training and configure failure recovery.
- **Ray Data (`ray.data`)** to provide the training and validation datasets that Ray Train shards and iterates efficiently. Ray Train uses Ray Data internally to support features like **mid-epoch resumption**.

These imports set up the building blocks you use later to overlap training with I/O and validation, and to resume training with minimal wasted work after failures.

In [ ]:
# ------------------------------------------------------------
# Standard library utilities
# ------------------------------------------------------------
import os            # File system paths and environment setup
import time          # Timing and performance measurements
import tempfile      # Temporary directories for local checkpoints
from typing import Dict, Any, Optional, Tuple  # Type hints for clarity

# ------------------------------------------------------------
# Data handling and preprocessing
# ------------------------------------------------------------
import numpy as np
import pandas as pd
from PIL import Image  # Image conversion for MNIST preprocessing

# ------------------------------------------------------------
# PyTorch and torchvision (model, training, transforms)
# ------------------------------------------------------------
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.datasets import MNIST
from torchvision.models import resnet18
from torchvision.transforms import Compose, Normalize, ToTensor

# ------------------------------------------------------------
# Ray Train and Ray Data
# ------------------------------------------------------------
import ray
import ray.data as rd                # Datasets used by Ray Train for streaming + recovery
from ray import train                # Training context and reporting APIs
from ray.train import (
    Checkpoint,                      # Persistent training state
    CheckpointUploadMode,            # Enable asynchronous checkpoint uploads
    RunConfig,                       # Experiment-level configuration
    ScalingConfig,                   # Distributed training resources
    FailureConfig,                   # Automatic failure recovery
)
from ray.train.torch import TorchTrainer  # Distributed PyTorch trainer
import ray.train.torch               # Device placement and DDP helpers

## Configure Shared Storage

You configure a **shared storage layout** that all Ray workers can access, regardless of which node they run on. In a multi-node Ray cluster, workers cannot reliably read from or write to the local filesystem of the head node, so you must use shared storage. Here you use mounted cluster storage. See the documentation for a list of [all available storage options](https://docs.anyscale.com/configuration/storage) that Anyscale offers.

In this tutorial, you use shared storage for three purposes:

- **Raw data storage** for downloading MNIST.
- **Parquet datasets** consumed by Ray Data during training and validation.
- **Training artifacts**, including checkpoints, metrics, and validation results produced by Ray Train.

Using a shared storage path is required for:
- Distributed data loading with Ray Data,
- Asynchronous checkpoint uploads,
- Failure recovery and mid-epoch resumption.

The paths you define here are reused throughout the rest of the notebook.

In [ ]:
# Root directory for all shared artifacts in this tutorial
SHARED_ROOT = "/mnt/cluster_storage"

# Location for raw dataset downloads (e.g., MNIST)
DATA_ROOT = os.path.join(SHARED_ROOT, "data")

# Location where the MNIST dataset is written as Parquet and later streamed by Ray Data
PARQUET_ROOT = os.path.join(SHARED_ROOT, "ray_train_async_tutorial", "mnist_parquet")


# Location where Ray Train stores run metadata, checkpoints, and validation results
RUN_STORAGE = os.path.join(SHARED_ROOT, "ray_train_async_tutorial", "runs")

# Ensure all required directories exist before training starts
os.makedirs(DATA_ROOT, exist_ok=True)
os.makedirs(PARQUET_ROOT, exist_ok=True)
os.makedirs(RUN_STORAGE, exist_ok=True)

# Print paths for visibility and debugging
print("DATA_ROOT:", DATA_ROOT)
print("PARQUET_ROOT:", PARQUET_ROOT)
print("RUN_STORAGE:", RUN_STORAGE)

## Write MNIST to Parquet with Stable Row IDs

You convert the MNIST training dataset into a **Parquet-backed Ray Data dataset** that Ray Train can stream efficiently during training.

The key requirement for this tutorial is **determinism**. To support mid-epoch resumption, you must be able to:
- Split the dataset deterministically into training and validation sets, and
- Restore the data iterator to an exact point in the epoch after a failure.

To enable this, you write a **stable, per-row `id` column** into the Parquet files. Ray Train uses this identifier internally to track which rows have already been consumed, allowing it to resume data iteration from the middle of an epoch instead of restarting from the beginning.

You write the dataset once to shared storage and reuse it across training runs.

In [ ]:
def write_mnist_parquet(parquet_root: str) -> None:
    # ------------------------------------------------------------
    # Write MNIST to Parquet for Ray Data streaming
    # ------------------------------------------------------------
    # This function converts the MNIST training dataset into a
    # Parquet dataset stored on shared storage.
    #
    # A stable per-row "id" column is required for mid-epoch
    # resumption so Ray Train can restore the dataset iterator
    # state after failures.
    # ------------------------------------------------------------
    
    # Avoid re-writing if already present
    if os.path.exists(parquet_root) and len(os.listdir(parquet_root)) > 0:
        print(f"Parquet already exists at: {parquet_root}")
        return

    print("Downloading MNIST...")
    ds = MNIST(root=DATA_ROOT, train=True, download=True)

    # Construct a DataFrame with a deterministic row id
    # - "id": stable identifier for each example
    # - "image": raw 28x28 uint8 arrays
    # - "label": integer class label
    df = pd.DataFrame(
        {
            "id": np.arange(len(ds), dtype=np.int64),
            "image": ds.data.numpy().tolist(),   # 28x28 uint8 arrays
            "label": ds.targets.numpy().astype(np.int64),
        }
    )

    # Write the dataset to Parquet on shared storage
    # Ray Data will later stream this data into Ray Train workers
    rd.from_pandas(df).write_parquet(parquet_root)
    print(f"Wrote MNIST Parquet to: {parquet_root}")

## Create Deterministic Train and Validation Datasets

You create training and validation datasets from the Parquet-backed MNIST data in a way that preserves **determinism and resumability**.

For mid-epoch resumption to work correctly, Ray Train must be able to:
- Track which rows have already been consumed in the current epoch, and
- Reconstruct the remaining portion of the dataset after a failure.

To guarantee this, you:
- **Avoid non-deterministic dataset operations** such as `random_shuffle()` or index-based splitting.
- Perform the train/validation split using a **stable, map-based filter** on the row `id`.
- Preserve the `id` column through all preprocessing steps.

This approach ensures that Ray Train can safely checkpoint and restore the dataset iterator state, even when training resumes from the middle of an epoch.

In [ ]:
def get_train_val_datasets(parquet_root: str, val_frac: float = 0.1) -> Tuple[rd.Dataset, rd.Dataset]:
    # ------------------------------------------------------------
    # Load Parquet dataset and create deterministic train/val split
    # ------------------------------------------------------------
    # For mid-epoch resumption, avoid non-map dataset operations
    # such as random_shuffle() or split_at_indices().
    # ------------------------------------------------------------
    
    # Read the Parquet dataset from shared storage
    base = rd.read_parquet(parquet_root)

    # Enforce a fixed validation fraction for this tutorial
    # For val_frac = 0.1, every 10th example goes to validation
    # This split is deterministic and implemented via filter()
    if abs(val_frac - 0.1) > 1e-9:
        raise ValueError(
            "For the mid-epoch resumption tutorial, keep val_frac=0.1 "
            "(deterministic split via id % 10)."
        )

    # Deterministic train/validation split based on the stable row id
    val_ds = base.filter(lambda r: (int(r["id"]) % 10) == 0)
    train_ds = base.filter(lambda r: (int(r["id"]) % 10) != 0)

    # Image preprocessing applied on-the-fly during streaming
    transform = Compose([ToTensor(), Normalize((0.5,), (0.5,))])

    def preprocess(row: Dict[str, Any]) -> Dict[str, Any]:
        # --------------------------------------------------------
        # Per-row preprocessing function
        # --------------------------------------------------------
        # IMPORTANT:
        # - Preserve the "id" column for iterator state tracking
        # - Apply transforms without changing row ordering
        # --------------------------------------------------------
        arr = np.array(row["image"], dtype=np.uint8)
        row["image"] = transform(Image.fromarray(arr))
        row["label"] = int(row["label"])
        row["id"] = int(row["id"])
        return row

    # Apply preprocessing lazily via map()
    # Ray Data streams these rows into training and validation workers
    return train_ds.map(preprocess), val_ds.map(preprocess)

## Create Streaming Train and Validation Datasets

You now write the raw MNIST dataset to Parquet on shared storage and define **streaming** training and validation datasets using Ray Data.

At this stage, you:
- Persist the raw dataset once so all workers can read from a stable, shared source.
- Define deterministic train and validation dataset pipelines.
- Keep preprocessing **lazy**, so data is transformed and streamed only when training begins.

Printing the datasets lets you verify that Ray Data has constructed the expected dataset graphs, without eagerly executing or caching the full dataset.

In [ ]:
# Write the raw MNIST dataset to Parquet on shared storage.
# This creates a stable, reusable data source for streaming.
# The write is idempotent and skipped if the data already exists.
write_mnist_parquet(PARQUET_ROOT)

# Define streaming training and validation datasets.
# These Ray Data pipelines are lazy and will only execute
# as data is consumed by Ray Train during training.
train_ds, val_ds = get_train_val_datasets(PARQUET_ROOT, val_frac=0.1)

# Print dataset summaries to confirm structure 
# without executing the full preprocessing pipeline.
print(train_ds)
print(val_ds)

## Define the Model

You define a simple ResNet-18 model adapted for MNIST. The model architecture is intentionally straightforward so you can focus on **training-system behavior** and not model complexity, when evaluating asynchronous checkpointing, asynchronous validation, and mid-epoch resumption.

The only modification you make is adjusting the first convolutional layer to accept **single-channel grayscale images** instead of RGB. Everything else remains standard and familiar.

In [ ]:
def build_model() -> nn.Module:
    # ------------------------------------------------------------
    # Build a ResNet-18 model adapted for MNIST
    # ------------------------------------------------------------
    # The model is intentionally simple so the tutorial focuses
    # on Ray Train features rather than model architecture.
    # ------------------------------------------------------------

    # Standard ResNet-18 with 10 output classes
    model = resnet18(num_classes=10)

    # Replace the first convolution to accept 1-channel
    # grayscale MNIST images instead of 3-channel RGB
    model.conv1 = nn.Conv2d(
        in_channels=1,
        out_channels=64,
        kernel_size=(7, 7),
        stride=(2, 2),
        padding=(3, 3),
        bias=False,
    )
    return model

## Define a Validation-Only Training Function

You define a **validation-only training function** that Ray Train runs asynchronously whenever a checkpoint is reported from training.

Instead of performing validation inline on the training workers, you:
- Load a reported checkpoint,
- Run evaluation on a separate set of GPU resources,
- And report validation metrics back to Ray Train.

This design enables **asynchronous validation**:
- Training continues uninterrupted on the main workers.
- Validation runs out of band in a separate Ray task.
- You can place validation on cheaper or smaller GPUs than training.

In this tutorial, you use a dedicated TorchTrainer for validation to reuse Ray Train’s distributed data loading and metric reporting APIs. While not shown in this tutorial, you may replace this seperate torch trainer with Ray actors that perform batch inference.

In [ ]:
def eval_only_train_fn(cfg: Dict[str, Any]) -> None:
    """
    Validation-only Ray Train function.

    This function loads a checkpoint, runs evaluation on the
    validation dataset, and reports metrics back to Ray Train.
    It does not perform any training or checkpoint uploads.
    """

    # Extract the checkpoint produced by training
    checkpoint: Checkpoint = cfg["checkpoint"]

    # Select the correct device for this validation worker
    device = ray.train.torch.get_device()

    # Build the model and switch to evaluation mode
    model = build_model().to(device).eval()

    # Load model weights from the checkpoint directory
    with checkpoint.as_directory() as ckpt_dir:
        state = torch.load(os.path.join(ckpt_dir, "model.pt"), map_location="cpu")
        model.load_state_dict(state)

    # Retrieve the validation dataset shard assigned to this worker
    ds_shard = train.get_dataset_shard("validation")

    # Create a streaming batch iterator over the validation data
    dl = ds_shard.iter_torch_batches(
        batch_size=cfg.get("batch_size", 256),
        dtypes={"image": torch.float32, "label": torch.long, "id": torch.long},
        device=device,
    )

    # Accumulate local accuracy metrics on each worker
    local_correct = 0
    local_total = 0
    with torch.no_grad():
        for batch in dl:
            x = batch["image"]
            y = batch["label"]
            logits = model(x)
            pred = logits.argmax(dim=1)
            local_correct += (pred == y).sum().item()
            local_total += y.numel()

    # Aggregate metrics across all validation workers
    correct_t = torch.tensor(local_correct, device=device, dtype=torch.long)
    total_t = torch.tensor(local_total, device=device, dtype=torch.long)

    if torch.distributed.is_available() and torch.distributed.is_initialized():
        torch.distributed.all_reduce(correct_t, op=torch.distributed.ReduceOp.SUM)
        torch.distributed.all_reduce(total_t, op=torch.distributed.ReduceOp.SUM)

    acc = (correct_t.float() / total_t.clamp_min(1).float()).item()

    # Report metrics without uploading a checkpoint
    # All workers must call report but only rank 0 reports metrics
    placeholder = Checkpoint(
        train.get_context().get_storage().build_checkpoint_path_from_name("placeholder")
    )

    if train.get_context().get_world_rank() == 0:
        train.report(
            metrics={"val_accuracy": acc},
            checkpoint=placeholder,
            checkpoint_upload_mode=CheckpointUploadMode.NO_UPLOAD,
        )
    else:
        train.report(metrics={}, checkpoint=None)

## Launch Asynchronous Validation Runs

You define a `validate_fn` that Ray Train invokes **asynchronously** whenever training reports a checkpoint with `validate_fn` and `validate_config`.

Instead of running validation inline on the training workers, Ray Train:
- Picks up the reported checkpoint,
- Launches this function as a separate Ray task,
- And runs validation in an independent `TorchTrainer`.

This design lets you:
- Keep training GPUs fully utilized,
- Run validation concurrently with training,
- And place validation on different (often cheaper) hardware.

In this tutorial, you configure validation to run on a single T4 GPU, but the same pattern generalizes to multi-GPU or distributed validation.

In [ ]:
def validate_fn(checkpoint: Checkpoint, cfg: Dict[str, Any]) -> Dict[str, Any]:
    """
    Asynchronous validation entry point.

    Ray Train launches this function as a Ray task when rank 0
    reports a checkpoint with validate_fn / validate_config.
    """

    # ------------------------------------------------------------
    # Name the validation run
    # ------------------------------------------------------------
    # Use a descriptive run name so validation jobs are easy
    # to identify in the Ray Train results and UI.
    run_name = f"{cfg['train_run_name']}_val_epoch_{cfg['epoch']}"

    # ------------------------------------------------------------
    # Configure a separate TorchTrainer for validation
    # ------------------------------------------------------------
    validator = TorchTrainer(
        train_loop_per_worker=eval_only_train_fn,      # Validation-only training loop
        train_loop_config={                            # Pass the checkpoint and validation-specific config
            "checkpoint": checkpoint,
            "batch_size": cfg.get("batch_size", 256),
        },
        datasets={"validation": cfg["dataset"]},       # Provide the validation dataset
        scaling_config=ScalingConfig(                  # Resource configuration for validation
            num_workers=1,
            use_gpu=True,
            accelerator_type="T4",                     # Place validation on a specific, cheaper GPU type
        ),
        run_config=RunConfig(name=run_name),           # Run configuration for this validation job
    )

    # ------------------------------------------------------------
    # Execute validation asynchronously
    # ------------------------------------------------------------
    # Training continues while this runs in parallel.
    res = validator.fit()

    # Return metrics so Ray Train can associate them
    # with the originating training checkpoint.
    return res.metrics

## Implement the Training Loop with Mid-Epoch Checkpoints and Async Validation

You now implement the core Ray Train training loop. This is where you combine all three advanced features in a single end-to-end workflow:

### 1) Mid-epoch resumption
You checkpoint and restore not only the model and optimizer state, but also the **Ray Data iterator state**. When a worker restarts, you resume from the middle of an epoch instead of restarting the epoch or skipping batches.

### 2) Asynchronous checkpointing
You upload checkpoints asynchronously by reporting them with `checkpoint_upload_mode=CheckpointUploadMode.ASYNC`. This allows training to continue while Ray uploads checkpoint data in the background.

### 3) Asynchronous validation
At the end of each epoch, you trigger validation by passing `validate_fn` and `validate_config` to `train.report()`. Ray Train launches validation as a separate Ray task, so training does not block on validation.

### How you structure the loop
You split responsibilities across workers:
- All workers run the forward/backward/update steps.
- Only rank 0 writes checkpoint files and attaches the checkpoint to `train.report()`.
- All workers call `train.report()` to keep the distributed workers synchronized.

You also log a stable progress marker (`last_id` and `global_step`) so you can prove that recovery resumes from the correct point in the data stream.

In [ ]:
def train_loop_per_worker(cfg: Dict[str, Any]) -> None:
    # ------------------------------------------------------------
    # NOTE: Imports inside the train loop keep the function self-
    # contained for Ray’s worker processes.
    # ------------------------------------------------------------
    import os, time, tempfile
    from typing import Any, Dict, Optional

    import torch
    import torch.nn as nn
    import torch.optim as optim

    import ray
    from ray import train
    from ray.train import Checkpoint

    # ------------------------------------------------------------
    # Ray Train context (rank / world size)
    # ------------------------------------------------------------
    ctx = train.get_context()
    rank = ctx.get_world_rank()
    world = ctx.get_world_size()

    # Select device for this worker (GPU if available/configured)
    device = ray.train.torch.get_device()

    # ------------------------------------------------------------
    # Model + optimizer (DDP-ready)
    # ------------------------------------------------------------
    model = build_model()

    # Wrap model for distributed training and move it to the right device.
    # Ray Train sets up the distributed process group and DDP wiring.
    model = ray.train.torch.prepare_model(model)  # DDP-wrapped

    # Access the underlying nn.Module when DDP wraps it.
    module = model.module if hasattr(model, "module") else model 

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=cfg["lr"]) 

    # ------------------------------------------------------------
    # Batch sizing
    # ------------------------------------------------------------
    # You specify a global batch size, then split it evenly across workers.
    global_batch_size = int(cfg["global_batch_size"])
    assert global_batch_size % world == 0, "global_batch_size must be divisible by num_workers"
    per_worker_batch = global_batch_size // world

    # Mid-epoch checkpoint cadence (in batches)
    ckpt_every_n_batches = int(cfg.get("ckpt_every_n_batches", 50))
    assert ckpt_every_n_batches > 0

    epochs = int(cfg["epochs"])

    # ------------------------------------------------------------
    # Resume state (mid-epoch capable)
    # ------------------------------------------------------------
    start_epoch = 0
    
    # If present, you apply this iterator state to the dataset shard once.
    resume_data_state: Optional[Dict[str, Any]] = None
    restored = False

    # Stable counters for debugging and proving resume correctness
    global_step = 0

    # Ray Train provides the latest checkpoint (if resuming a run).
    loaded_ckpt = train.get_checkpoint()

    meta: Dict[str, Any] = {}

    if loaded_ckpt is not None:
        restored = True
        with loaded_ckpt.as_directory() as d:
            model_path = os.path.join(d, "model.pt")
            opt_path = os.path.join(d, "optim.pt")
            meta_path = os.path.join(d, "meta.pt")
            data_path = os.path.join(d, "data_iter.pt")

            # Restore model and optimizer state (standard fault tolerance)
            if os.path.exists(model_path):
                module.load_state_dict(torch.load(model_path, map_location="cpu"))
            if os.path.exists(opt_path):
                optimizer.load_state_dict(torch.load(opt_path, map_location="cpu"))

            # Restore metadata (epoch, global_step, last_id for debug)
            if os.path.exists(meta_path):
                meta = torch.load(meta_path, map_location="cpu") or {}

            # If you checkpointed data iterator state, you can resume mid-epoch.
            has_data_iter = os.path.exists(data_path)
            if has_data_iter:
                # Mid-epoch resume: restart in the SAME epoch and restore iterator state.
                resume_data_state = torch.load(data_path, map_location="cpu")
                start_epoch = int(meta.get("epoch", 0))
            else:
                # Epoch-boundary resume: continue at the NEXT epoch.
                start_epoch = int(meta.get("epoch", 0)) + 1

            global_step = int(meta.get("global_step", 0))

            # Print resume details from rank 0 for debugging
            if rank == 0:
                print("\n===== RESUME DEBUG =====")
                print("checkpoint_dir:", d)
                print("meta:", meta)
                print("has data_iter.pt:", has_data_iter)
                print("computed start_epoch:", start_epoch)
                print("global_step:", global_step)
                print("========================\n")

    # ------------------------------------------------------------
    # Train epochs
    # ------------------------------------------------------------
    for epoch in range(start_epoch, epochs):
        model.train()

        is_resumed_epoch = resume_data_state is not None
        if rank == 0:
            print(f"[rank0] entering epoch={epoch} restored={restored} is_resumed_epoch={is_resumed_epoch}")

        # --------------------------------------------------------
        # Create the dataset shard (with optional restored state)
        # --------------------------------------------------------
        # IMPORTANT: If you resume mid-epoch, you must apply the same
        # iterator state on ALL ranks.
        if resume_data_state is not None:
            shard = train.get_dataset_shard("train", state_dict=resume_data_state)
            if rank == 0:
                print(f"[rank0] Applied resume_data_state at epoch={epoch}")
            resume_data_state = None  # apply once
        else:
            shard = train.get_dataset_shard("train")

        # Create a streaming iterator over the shard
        dl = shard.iter_torch_batches(
            batch_size=per_worker_batch,
            dtypes={"image": torch.float32, "label": torch.long, "id": torch.long},
            device=device,
            prefetch_batches=int(cfg.get("prefetch_batches", 2)),
        )

        # Track per-epoch timing and last observed values for reporting
        t0 = time.time()
        last_loss = None
        last_id = None
        batch_idx = -1

        # --------------------------------------------------------
        # Batch loop
        # --------------------------------------------------------
        for batch_idx, batch in enumerate(dl):
            if rank == 0 and batch_idx == 0:
                ids0 = batch["id"][:8].detach().cpu().tolist()
                print(f"[rank0] epoch={epoch} FIRST batch ids[:8]={ids0} (global_step={global_step})")

            x = batch["image"]
            y = batch["label"]

            logits = model(x)
            loss = criterion(logits, y)

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

            last_loss = loss.detach()
            global_step += 1

            # Record a deterministic progress marker from the data stream
            last_id = int(batch["id"][-1].detach().cpu().item())

            # ----------------------------------------------------
            # Mid-epoch checkpointing (model + optim + data state)
            # ----------------------------------------------------
            # IMPORTANT: All ranks must call shard.state_dict() at the same time.
            if (batch_idx + 1) % ckpt_every_n_batches == 0:
                data_state_dict = shard.state_dict()

                if rank == 0:
                    # Write checkpoint files locally; Ray Train handles persistence.                    
                    local_ckpt_dir = tempfile.mkdtemp(prefix="ray_train_ckpt_")
                    torch.save(module.state_dict(), os.path.join(local_ckpt_dir, "model.pt"))
                    torch.save(optimizer.state_dict(), os.path.join(local_ckpt_dir, "optim.pt"))
                    torch.save(
                        {
                            "epoch": epoch,
                            "batch_idx": batch_idx,      # debug only (not used to resume)
                            "global_step": global_step,  # stable across restart
                            "last_id": last_id,          # stable proof-of-position
                        },
                        os.path.join(local_ckpt_dir, "meta.pt"),
                    )
                    # Save iterator state for mid-epoch resumption
                    torch.save(data_state_dict, os.path.join(local_ckpt_dir, "data_iter.pt"))

                    # Report metrics + checkpoint. ASYNC upload avoids blocking training.
                    train.report(
                        metrics={
                            "epoch": epoch,
                            "batch_idx": batch_idx,
                            "global_step": global_step,
                            "last_id": last_id,
                            "train_loss": float(last_loss.item()) if last_loss is not None else None,
                        },
                        checkpoint=Checkpoint.from_directory(local_ckpt_dir), 
                        checkpoint_upload_mode=CheckpointUploadMode.ASYNC,
                    )
                else:
                    # Barrier: keep report call count aligned across workers.
                    train.report(metrics={}, checkpoint=None)

        # --------------------------------------------------------
        # End-of-epoch reporting + async validation trigger
        # --------------------------------------------------------
        step_time_s = time.time() - t0
        metrics = {
            "epoch": epoch,
            "global_step": global_step,
            "last_id": last_id,
            "train_loss": float(last_loss.item()) if last_loss is not None else None,
            "epoch_time_s": step_time_s,
        }

        # Save iterator state again so the end-of-epoch checkpoint
        # can also resume mid-epoch if a failure happens early next epoch.
        data_state_dict = shard.state_dict()

        if rank == 0:
            local_ckpt_dir = tempfile.mkdtemp(prefix="ray_train_ckpt_")
            torch.save(module.state_dict(), os.path.join(local_ckpt_dir, "model.pt"))
            torch.save(optimizer.state_dict(), os.path.join(local_ckpt_dir, "optim.pt"))
            torch.save(
                {
                    "epoch": epoch,
                    "batch_idx": batch_idx,
                    "global_step": global_step,
                    "last_id": last_id,
                },
                os.path.join(local_ckpt_dir, "meta.pt"),
            )
            torch.save(data_state_dict, os.path.join(local_ckpt_dir, "data_iter.pt"))

            # Report end-of-epoch checkpoint with:
            # - ASYNC upload to avoid blocking training
            # - validate_fn to kick off async validation
            train.report(
                metrics=metrics,
                checkpoint=Checkpoint.from_directory(local_ckpt_dir),
                checkpoint_upload_mode=CheckpointUploadMode.ASYNC,
                validate_fn=validate_fn,
                validate_config={
                    "dataset": cfg["validation_dataset"],
                    "train_run_name": ctx.get_experiment_name(),
                    "epoch": epoch,
                    "batch_size": int(cfg.get("val_batch_size", 256)),
                },
            )
        else:
            # Barrier: keep report call count aligned across workers.
            train.report(metrics={}, checkpoint=None)

## Configure the TorchTrainer

You now configure `TorchTrainer` to run distributed training and enable the advanced features you implemented in the training loop.

### What you configure here
- **Distributed training resources**: You run training with two GPU workers.
- **Mid-epoch checkpoint cadence**: You checkpoint every `ckpt_every_n_batches` batches so you can resume from the middle of an epoch after failures.
- **Dataset checkpoint support**: You tell Ray Train which column uniquely identifies each row (`id_column="id"`). Ray Train uses this information to support mid-epoch resumption.
- **Async validation wiring**: You pass the validation dataset through the training config so the training loop can trigger `validate_fn` at the end of each epoch.
- **Failure recovery**: You allow the run to recover from failures by restarting workers up to `max_failures` times.

You also place training on an A10G GPU type. Later, you place validation on a different GPU type (T4) to demonstrate heterogeneous hardware.

In [ ]:
trainer = TorchTrainer(
    train_loop_per_worker=train_loop_per_worker,   # Training function executed on each training worker
    train_loop_config={                            # Configuration passed into your train_loop_per_worker
        "lr": 1e-4,
        "global_batch_size": 256,
        "epochs": 3,
        "prefetch_batches": 2,                      # Prefetching improves throughput for streaming batches
        "ckpt_every_n_batches": 50,                 # Mid-epoch checkpoint cadence (in batches)
        "val_batch_size": 256,                      # Validation configuration passed through to validate_fn
        "validation_dataset": val_ds,  
    },

    # Training dataset sharded across training workers
    datasets={"train": train_ds},  

    # ------------------------------------------------------------
    # Dataset checkpoint configuration (required for mid-epoch resume)
    # ------------------------------------------------------------
    # You provide a stable id column so Ray Train can track which rows
    # have already been yielded in the current epoch and restore the
    # iterator state after failures.
    dataset_config=train.DataConfig(
        dataset_checkpoint_configs={
            "train": train.DatasetCheckpointConfig(
                id_column="id",
            )
        }
    ),

    # ------------------------------------------------------------
    # Training resources (distributed DDP workers)
    # ------------------------------------------------------------
    scaling_config=ScalingConfig(
        num_workers=2,
        use_gpu=True,
        accelerator_type="A10G",                       # training runs on A10Gs
    ),

    # ------------------------------------------------------------
    # Run configuration (storage + failure recovery)
    # ------------------------------------------------------------
    run_config=RunConfig(
        name="mnist_async_ckpt_async_val",
        storage_path=RUN_STORAGE,                      # shared storage for checkpoints and results
        failure_config=FailureConfig(max_failures=1),
    ),
)

## Start Training

You now launch the distributed training job. Ray Train starts the training workers, streams data to them, uploads checkpoints asynchronously, and triggers validation runs in parallel as checkpoints are reported.

In [ ]:
# Launch the distributed training run
result = trainer.fit()

## Observe Mid-Epoch Resumption in Action

If training stops early (for example due to a worker failure, preemption, or manual interruption), Ray Train can resume **from the middle of the epoch**, and is not limited to started up again only from the beginning.

When training resumes, Ray Train:
- Restores the model and optimizer state from the latest checkpoint.
- Restores the Ray Data iterator state.
- Continues training from the **next unseen rows** in the dataset.

The log example output below shows this behavior clearly:
- Training resumes at the same epoch.
- The dataset iterator state is restored.
- Training continues from the correct position in the data stream instead of restarting the epoch.

![Mid-epoch resumption debug output](images/midepoch_image.png)

## Inspect Training and Validation Metrics

You can now inspect the metrics produced by the training run. Because validation runs asynchronously, validation metrics may appear **later** than the corresponding training metrics, but Ray Train correctly associates them with the checkpoint that triggered validation.

The metrics dataframe includes:
- Training metrics reported during and at the end of each epoch.
- Validation metrics reported asynchronously by the validation runs.
- Metadata such as epoch, batch index, and global step that help you correlate events.

### How to read this table

In this tutorial, you report two kinds of checkpoints, and you can distinguish them by `batch_idx`:

- Rows with a **non-null `batch_idx`** (for example 49, 99, 149, 199) are **mid-epoch checkpoints**. You emit these every `ckpt_every_n_batches` to support mid-epoch resumption.
- Rows with **`batch_idx = NaN`** are **end-of-epoch reports**. These rows include `epoch_time_s` and are the ones where you also trigger **asynchronous validation**.

In [ ]:
# Training + validated metrics (validation metrics may arrive async but are associated with checkpoints)
df = result.metrics_dataframe
df.tail(20)

## Load the Final Checkpoint and Run Inference

You finish by loading the **final Ray Train checkpoint** and running a quick inference sanity check on MNIST test images.

This step validates the full end-to-end workflow:
- You trained with asynchronous checkpoint uploads.
- You triggered validation asynchronously during training.
- You persisted checkpoints to shared storage.
- You can load the final checkpoint and use it like a normal PyTorch model.

You then visualize a few predictions so you can quickly confirm the model learned a reasonable digit classifier.

Note that at scale, you would perform inference with Ray Data for efficency, but for this tutorial, the following will suffice.

In [ ]:
import matplotlib.pyplot as plt

# Load the final checkpoint produced by trainer.fit() (if available)
if result.checkpoint:
    # Materialize the checkpoint contents into a local directory for reading
    with result.checkpoint.as_directory() as d:
        # Load model weights saved by the training loop
        model_path = os.path.join(d, "model.pt")

        model = build_model()
        model.load_state_dict(torch.load(model_path, map_location="cpu"))
        model.eval()

        # Prepare MNIST test set for inference
        test = MNIST(root=DATA_ROOT, train=False, download=True)
        transform = Compose([ToTensor(), Normalize((0.5,), (0.5,))])

        # Visualize a small batch of predictions
        num_images = 10
        fig, axs = plt.subplots(1, num_images, figsize=(22, 4))  # taller

        with torch.no_grad():
            for i in range(num_images):
                img, label = test[i]

                # Apply the same normalization used during training
                img_tensor = transform(img).unsqueeze(0)

                # Predict digit class
                pred = model(img_tensor).argmax(dim=1).item()

                # Plot image + prediction
                axs[i].imshow(img, cmap="gray")
                axs[i].axis("off")
                axs[i].set_title(
                    f"Pred: {pred}\nTrue: {label}",
                    fontsize=13,        # larger text
                    pad=6               # spacing from image
                )

        # Add a figure title and tighten layout for cleaner display
        fig.suptitle(
            "Ray Train final checkpoint - MNIST predictions",
            fontsize=18,
            y=1.02
        )
        fig.tight_layout(rect=[0, 0, 1, 0.9])
        plt.show()

## (Optional) Clean Up Tutorial Artifacts

If you’re finished experimenting, you can remove the data, checkpoints, and run artifacts created by this tutorial.

This step is optional, but useful if you want to:
- Re-run the tutorial from a clean state.
- Free up shared storage space.
- Avoid mixing results from multiple runs.

In [ ]:
import shutil
from pathlib import Path

path = Path("/mnt/cluster_storage/ray_train_async_tutorial")

if path.exists():
    shutil.rmtree(path)
    print(f"Deleted directory: {path}")
else:
    print(f"Directory does not exist: {path}")


## Conclusion

In this tutorial, you built an end-to-end distributed training workflow that focuses on **eliminating training bottlenecks**, not just scaling model execution.

You learned how to:
- **Stream data with Ray Data** so GPUs stay fed without eagerly materializing datasets.
- **Checkpoint asynchronously** to avoid blocking training on storage I/O.
- **Run validation asynchronously** on separate resources so training continues uninterrupted.
- **Resume from the middle of an epoch** by checkpointing and restoring both model state and data iterator state.

Together, these features let you keep GPUs busy, reduce wasted work after failures, and use heterogeneous hardware more efficiently. Instead of treating checkpointing, validation, and failure recovery as unavoidable pauses, you move them off the critical training path.

While this tutorial uses MNIST for simplicity, the same patterns apply to (and shine best with) large-scale workloads:
- Long-running training jobs with frequent checkpoints,
- Expensive GPU clusters where idle time is costly,
- Preemptible or spot instances where failures are expected,
- And validation workloads that do not need the same resources as training.

By combining Ray Train and Ray Data, you can build distributed training systems that are not only scalable, but also **efficient, resilient, and production-ready**.